## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [2]:
MY_BUBBLE_FILE = "anti_sistem.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_sistem
Texte: 50


,id,agent,text
0,yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg,Anti-sistem,Semneaza Bo$$ ca la urmatoarele alegerii nu ma...
1,yt_pO0JOPX1I7Q_UgyExKkzQjU2eOVTG7J4AaABAg,Anti-sistem,ESTE NEVOIE DE O FESTAÑIE LA TOATE NIVELURILE ...
2,yt_6_Hc2S02Duw_Ugz2UatUIFNpL1SP7u54AaABAg,Anti-sistem,Orcii fac ore suplimentare!! Bravo! Daca ati m...
3,yt_YFbJhBc_9jo_Ugx9GFXKkTHUqa4NYcZ4AaABAg,Anti-sistem,Câtă nesimțire!!! Câtă hoție pe fațaă!!! Ce ră...
4,yt_YFbJhBc_9jo_UgzrK_gcZVEbQXhfL614AaABAg,Anti-sistem,"Biserica, aceasta sinecura de sifonat bani, es..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [3]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Semneaza Bo$$ ca la urmatoarele alegerii nu mai iesi presedinte. Noi ca tara si popor suntem rupti in cur cu salarii de vietnam si preturi de SIngapore.... dar ajutam cu banii Ukraina ... alta tara corupta la fel si Rusia


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [4]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|████

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 de texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea vectorilor embedded, adică 384.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [5]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_sistem
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: 50

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [6]:
# Text nou introdus în aplicație

input_text = "Guvernul României a căzut după o moțiune de cenzură care a pus stop austerității create de Bolojan, dar a creat o și mai mare instabilitate politică pentru că nimeni nu vrea să fie prim ministru."

In [7]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [8]:
# query_vector

In [9]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.587
Text: Scoateți instituțiile la treabă să sancționeze căci pe cetățeni dacă nu îi arzi la bani nu se modelează corespunzător. România este o țară ineficientă, infrastructura nu ne permite să economisim. Este cald afară dar nu pot circula cu bicicleta deoarece este foarte poluat. Măriți impozitul pe autoturism personal mai mult că acolo se duce mult din devalorizarea monezii.

Rezultat 2
Scor: 0.567
Text: Romania este falimentara . Nimeni nu o mai imprumuta . A fost refuzata. Băsescu dă verdictul care sperie România: 'Nu avem bani! Nu există marjă pentru ieftinirea carburanților ... România intră într-o zonă periculoasă din punct de vedere economic, avertizează fostul președinte Traian Băsescu, care susține că statul nu mai are nicio marjă reală de intervenție pentru a tempera explozia prețurilor la carburanți. Pregatiti-va de dezastru si nu mai cereti bani ..

Rezultat 3
Scor: 0.517
Text: psd și aur, împreună cu șefii magistraților, frânează sistematic România, ș

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

Toate rezultatele sunt foarte relevante, surpinzător.

Textele recuperate nu exrpima exact vocea agentului, încât nu a generat ceva, doar a dat copy paste la cele mai fitting comentarii din model, însă generativ mă aștept să poată să fie un match foarte bun.

Cred ca cel mai slab dintre toate o fi rezultatul 5 (conform scorului), dar și acela este relevant având în vedere discuțiile prezente pentru a forma un nou guvern.